# 122 — Focused real flow-step PCP evaluation

Evaluate up to 48 historically mixed **original validation** roots from the old idx10–29 cohort, excluding all 18 previous pilot source roots. The actual available count can be smaller (the old cohort has about 42 mixed validation roots before exclusions). No roots are duplicated or replaced with training roots. Selection uses the frozen historical nine-candidate outcomes, never new PCP results, and balances historical stock failure/success as availability allows.

The shared immutable manifest records every selected source/group ID, original split, excluded prior roots, checkpoint SHA, frozen snapshot, controls, and actual enrichment/counts. Three configurable shards share that manifest. This is an **enriched reused-validation comparison**, not independent policy evaluation or an estimate of population success rate.

Real PCP runs inside the sampler at existing zero-based step 3, s=.7, K=1: positive/negative Q-gradient and matched-RMS random controls at clean radii .02/.06, regenerated zero-correction P&P, historical stock, and one fresh P&P chunk (nine candidates per root, one shared future seed). Complete groups resume. Zero correction must exactly match the live P&P chunk and perturbation stream. The terminal-action critic is extrapolated to intermediate clean estimates.

The fixed late-fusion checkpoint SHA is verified before root work. Supply the same .pt manually through upload or Drive; checkpoints are never uploaded to Supabase. Default is a CPU manifest/gradient dry run. Set RUN_INTERVENTION=True and select a CUDA runtime only for simulator evaluation. ROOT_LIMIT=None covers all roots assigned to this shard; set 1 for a smoke test.


In [ ]:
SHARD_INDEX = 0  # 0, 1, or 2
ROOT_LIMIT = None  # full assigned shard; set 1 for a smoke test
RUN_INTERVENTION = False
CHECKPOINT_PATH = ''  # /content/uploaded.pt or /content/drive/MyDrive/.../checkpoint.pt
UPLOAD_CHECKPOINT = False  # Optional forced upload; an empty path also prompts automatically
MOUNT_DRIVE = False  # True for a Drive checkpoint
DEVICE = 'cpu'  # CPU is sufficient for the first benchmark
SHARD_COUNT = 3
TARGET_ROOTS = 48  # actual inventory may be smaller; never duplicated
EXPECTED_CHECKPOINT_SHA256 = '9b8800bc9f27116afcb1812f93f8a9f6767ac012c7cdac790502032fbb7d0c8e'


In [ ]:
# Install NVIDIA EGL only when simulator intervention is enabled, before MuJoCo imports.
if RUN_INTERVENTION:
    import re, subprocess
    driver = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'],
        text=True).splitlines()[0].strip()
    package = f"libnvidia-gl-{re.match(r'\d+', driver).group()}"
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    availability = subprocess.check_output(['apt-cache', 'policy', package], text=True)
    if 'Candidate: (none)' in availability:
        raise RuntimeError(f'{package} is unavailable in this runtime')
    subprocess.run(['apt-get', 'install', '-y', '--no-install-recommends', package], check=True)
    subprocess.run('ldconfig -p | grep libEGL_nvidia', shell=True, check=True)

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
if UPLOAD_CHECKPOINT or not CHECKPOINT_PATH:
    from google.colab import files
    from pathlib import Path
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one scalar critic .pt checkpoint.')
    CHECKPOINT_PATH = str(Path(next(iter(uploaded))).resolve())

EXTRAS = 'sim'
SETUP_ENV = RUN_INTERVENTION
REPO_REF = 'codex/smolvla-q-program'
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/codex/smolvla-q-program/pnp-vla/scripts/colab_bootstrap.py').read().decode())


In [ ]:
from pathlib import Path
from pnp.smolvla_focused_pcp_eval import run_focused_flow_pcp_eval
if not CHECKPOINT_PATH:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one scalar critic .pt checkpoint.')
    CHECKPOINT_PATH = str(Path(next(iter(uploaded))).resolve())
if not Path(CHECKPOINT_PATH).is_file():
    raise FileNotFoundError(f'Checkpoint not found: {CHECKPOINT_PATH}. Supply a Colab/Drive path or clear CHECKPOINT_PATH to upload.')
assert 0 <= SHARD_INDEX < SHARD_COUNT
REPORT_PATH = f'/content/focused_flow_pcp_eval_shard_{SHARD_INDEX}.json'
report = run_focused_flow_pcp_eval(
    expected_checkpoint_sha=EXPECTED_CHECKPOINT_SHA256,
    target_roots=TARGET_ROOTS, shard_count=SHARD_COUNT,
    checkpoint_path=CHECKPOINT_PATH, shard_index=SHARD_INDEX,
    root_limit=ROOT_LIMIT, run_intervention=RUN_INTERVENTION,
    device=DEVICE, output_path=REPORT_PATH)
print({'report': REPORT_PATH, 'new_simulator_roots': report['new_roots']})


Review `selection_audit` and `cohort_manifest` before enabling interventions. The CPU dry run writes only the local report and downloads the selected frozen critic-input artifacts. On opt-in, the full cohort manifest is frozen in Supabase under a distinct focused experiment; every branch records original validation provenance and checkpoint identity.

Compare each ±Q/random arm with regenerated zero P&P at its radius, including harms as well as improvements. Report paired counts and the number of selected/eligible roots. Selection deliberately enriches for historically mixed outcomes; generalization and population success-rate claims require a fresh cohort.
